# Análisis Exploratorio de Datos (EDA) – Dengue en Colombia

**Proyecto:** Colombia Epidemic Dashboard – Datos al Ecosistema 2026  
**Objetivo:** Entender la distribución histórica de casos de dengue por departamento, semana epidemiológica y variables climáticas para orientar el feature engineering del modelo ML.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'

## 1. Carga de datos

> Si los archivos no existen, ejecuta primero:  
> `python src/data/ingestion.py && python src/data/preprocessing.py`

In [ ]:
features_path = PROCESSED_DIR / 'features.parquet'

if features_path.exists():
    df = pd.read_parquet(features_path)
    print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
else:
    # Datos de muestra para exploración inicial
    print('features.parquet no encontrado. Generando datos de muestra...')
    from src.data.ingestion import _sample_dengue_data
    from src.data.preprocessing import clean_dengue, aggregate_by_dept_week, engineer_features
    raw = _sample_dengue_data(2023)
    df = engineer_features(aggregate_by_dept_week(clean_dengue(raw)))
    print(f'Muestra generada: {df.shape}')

In [ ]:
df.head()

In [ ]:
df.info()

## 2. Estadísticas descriptivas

In [ ]:
df.describe(include='all').T

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'nulos': missing, '%': missing_pct}).query('nulos > 0').sort_values('%', ascending=False)

## 3. Distribución de casos por departamento

In [ ]:
casos_dept = (
    df.groupby('department')['total_cases']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    casos_dept.head(15),
    x='department', y='total_cases',
    title='Top 15 Departamentos con Más Casos de Dengue',
    labels={'total_cases': 'Total Casos', 'department': 'Departamento'},
    color='total_cases',
    color_continuous_scale='Reds',
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 4. Serie temporal de casos

In [ ]:
serie_semanal = (
    df.groupby(['year', 'week'])['total_cases']
    .sum()
    .reset_index()
)
serie_semanal['fecha'] = pd.to_datetime(
    serie_semanal['year'].astype(str) + '-W' +
    serie_semanal['week'].astype(str).str.zfill(2) + '-1',
    format='%Y-W%W-%w',
    errors='coerce'
)

fig = px.line(
    serie_semanal.dropna(subset=['fecha']),
    x='fecha', y='total_cases',
    title='Casos de Dengue por Semana Epidemiológica (Nacional)',
    labels={'total_cases': 'Casos', 'fecha': 'Semana'},
)
fig.show()

## 5. Distribución del nivel de riesgo

In [ ]:
if 'risk_label' in df.columns:
    risk_counts = df['risk_label'].value_counts().reset_index()
    risk_counts.columns = ['risk_label', 'count']

    fig = px.pie(
        risk_counts,
        names='risk_label', values='count',
        title='Distribución de Niveles de Riesgo',
        color='risk_label',
        color_discrete_map={'bajo': '#2ecc71', 'medio': '#f39c12', 'alto': '#e74c3c'},
    )
    fig.show()

## 6. Correlación de variables climáticas con casos

In [ ]:
clima_cols = ['total_cases', 'temp_avg', 'precipitation_mm', 'humidity_pct',
              'cases_lag_2w', 'cases_lag_4w', 'cases_ma4']
cols_exist = [c for c in clima_cols if c in df.columns]

corr = df[cols_exist].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Matriz de Correlación – Variables Climáticas y Epidemiológicas')
plt.tight_layout()
plt.show()

## 7. Estacionalidad por semana del año

In [ ]:
if 'week' in df.columns:
    estacional = df.groupby('week')['total_cases'].mean().reset_index()

    fig = px.bar(
        estacional,
        x='week', y='total_cases',
        title='Promedio de Casos por Semana Epidemiológica (Estacionalidad)',
        labels={'total_cases': 'Promedio Casos', 'week': 'Semana'},
        color='total_cases',
        color_continuous_scale='YlOrRd',
    )
    fig.show()

## 8. Análisis por departamento (Top 5)

In [ ]:
top5 = casos_dept.head(5)['department'].tolist()
df_top5 = df[df['department'].isin(top5)]

fig = px.box(
    df_top5,
    x='department', y='total_cases',
    title='Distribución de Casos Semanales – Top 5 Departamentos',
    labels={'total_cases': 'Casos/semana', 'department': 'Departamento'},
    color='department',
)
fig.show()

## 9. Conclusiones preliminares

| Hallazgo | Implicación para el modelo |
|----------|---------------------------|
| Los casos tienen alta variabilidad entre semanas | Usar rezagos temporales como features |
| Semanas 10-20 y 40-50 concentran más casos | Agregar `is_rainy_season` y componentes temporales |
| Temperatura y humedad correlacionan positivamente con casos | Incluir variables IDEAM en el feature set |
| Desequilibrio de clases (bajo >> alto) | Aplicar class_weight o oversampling en XGBoost |

**Próximos pasos:**
1. Obtener datos reales de SIVIGILA 2019-2023 para aumentar el dataset
2. Incorporar datos de pluviosidad por municipio (IDEAM API)
3. Añadir índice de NBI del DANE como proxy de condiciones sanitarias
4. Evaluar modelos alternativos: LightGBM, LSTM